In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("BigDataProject") \
    .config("spark.driver.host", "jupyter") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"
    ) \
    .enableHiveSupport() \
    .getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/spark/.ivy2/cache
The jars for the packages stored in: /home/spark/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c23e728b-e6de-4a69-b1c1-b0991ec484d6;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 378ms :: artifacts dl 21ms
	:: m

In [3]:
print(spark.version)
spark.range(5).show()

3.5.1


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [4]:
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "students") \
    .option("startingOffsets", "earliest") \
    .load()

df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)") \
    .writeStream \
    .format("console") \
    .outputMode("append") \
    .start() \
    .awaitTermination(20)

26/06/21 20:23:04 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-7e8d8e78-7cf7-47c8-a3e6-d574790d0c1e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/21 20:23:04 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/21 20:23:04 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+----+--------------------+
| key|               value|
+----+--------------------+
|NULL|{"id":1,"name":"A...|
|NULL|{"id":2,"name":"A...|
|NULL|{"id":1,"name":"A...|
|NULL|{"id":2,"name":"A...|
+----+--------------------+



False

In [5]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col, from_json

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType())
])

df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "students") \
    .option("startingOffsets", "earliest") \
    .load()

parsed_df = df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

query = parsed_df.writeStream \
    .format("parquet") \
    .option("path", "hdfs://hadoop-master:9000/user/spark/students_stream") \
    .option("checkpointLocation", "hdfs://hadoop-master:9000/user/spark/checkpoints/students_stream") \
    .outputMode("append") \
    .trigger(processingTime="10 seconds") \
    .start()

query.awaitTermination(30)

26/06/21 20:24:47 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/21 20:24:49 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


False

In [6]:
spark.sql("""
CREATE TABLE IF NOT EXISTS students_stream (id INT, name STRING)
USING PARQUET
LOCATION 'hdfs://hadoop-master:9000/user/spark/students_stream'
""")

DataFrame[]

In [7]:
spark.sql("SELECT * FROM default.students_stream LIMIT 10").show()

+---+-----+
| id| name|
+---+-----+
|  1|Ahmed|
|  2|  Ali|
|  1|Ahmed|
|  2|  Ali|
+---+-----+



In [8]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [2]:
spark.sql("SHOW DATABASES").show()
print(spark.conf.get("spark.sql.catalogImplementation"))  # لازم يطبع "hive"

26/06/21 20:55:32 WARN metastore: Failed to connect to the MetaStore Server...
26/06/21 20:55:33 WARN metastore: Failed to connect to the MetaStore Server...
26/06/21 20:55:34 WARN metastore: Failed to connect to the MetaStore Server...
26/06/21 20:55:35 WARN HiveClientImpl: HiveClient got thrift exception, destroying client and retrying (0 tries remaining)
org.apache.hadoop.hive.ql.metadata.HiveException: java.lang.RuntimeException: Unable to instantiate org.apache.hadoop.hive.ql.metadata.SessionHiveMetaStoreClient
	at org.apache.hadoop.hive.ql.metadata.Hive.getDatabase(Hive.java:1567)
	at org.apache.hadoop.hive.ql.metadata.Hive.databaseExists(Hive.java:1552)
	at org.apache.spark.sql.hive.client.Shim_v0_12.databaseExists(HiveShim.scala:609)
	at org.apache.spark.sql.hive.client.HiveClientImpl.$anonfun$databaseExists$1(HiveClientImpl.scala:406)
	at scala.runtime.java8.JFunction0$mcZ$sp.apply(JFunction0$mcZ$sp.java:23)
	at org.apache.spark.sql.hive.client.HiveClientImpl.$anonfun$withHive

AnalysisException: org.apache.hadoop.hive.ql.metadata.HiveException: java.lang.RuntimeException: Unable to instantiate org.apache.hadoop.hive.ql.metadata.SessionHiveMetaStoreClient

In [2]:
spark.sql("SHOW DATABASES").show()

26/06/21 21:03:39 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/06/21 21:03:39 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/06/21 21:03:57 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/06/21 21:03:57 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.21.0.2


+---------+
|namespace|
+---------+
|  default|
|  test_db|
+---------+



In [3]:
spark.sql("SELECT * FROM test_db.students").show()

26/06/21 21:07:01 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
26/06/21 21:07:03 WARN ShellBasedUnixGroupsMapping: unable to return groups for user hdfs
PartialGroupNameException The user name 'hdfs' is not found. id: ‘hdfs’: no such user
id: ‘hdfs’: no such user

	at org.apache.hadoop.security.ShellBasedUnixGroupsMapping.resolvePartialGroupNames(ShellBasedUnixGroupsMapping.java:294)
	at org.apache.hadoop.security.ShellBasedUnixGroupsMapping.getUnixGroups(ShellBasedUnixGroupsMapping.java:207)
	at org.apache.hadoop.security.ShellBasedUnixGroupsMapping.getGroups(ShellBasedUnixGroupsMapping.java:97)
	at org.apache.hadoop.security.JniBasedUnixGroupsMappingWithFallback.getGroups(JniBasedUnixGroupsMappingWithFallback.java:51)
	at org.apache.hadoop.security.Groups$GroupCacheLoader.fetchGroupList(Groups.java:387)
	at org.apache.hadoop.security.Groups$GroupCacheLoader.load(Groups.java:321)
	at org.apache.hadoop.security.Groups$GroupCacheLoader.load(Group

+---+-----+
| id| name|
+---+-----+
|  1|Ahmed|
|  2|  Ali|
+---+-----+



In [4]:
spark.sql("CREATE TABLE IF NOT EXISTS test_db.verify_session (x INT) USING PARQUET")
spark.sql("INSERT INTO test_db.verify_session VALUES (1)")
spark.sql("SELECT * FROM test_db.verify_session").show()

26/06/21 21:07:34 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/06/21 21:07:34 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/06/21 21:07:34 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


+---+
|  x|
+---+
|  1|
+---+



In [5]:
spark.sql("DESCRIBE DATABASE EXTENDED test_db").show(truncate=False)

+--------------+--------------------------------------------------------+
|info_name     |info_value                                              |
+--------------+--------------------------------------------------------+
|Catalog Name  |spark_catalog                                           |
|Namespace Name|test_db                                                 |
|Comment       |                                                        |
|Location      |hdfs://hadoop-master:9000/user/hive/warehouse/test_db.db|
|Owner         |hdfs                                                    |
|Properties    |                                                        |
+--------------+--------------------------------------------------------+

